# Ch8. Exponential Smoothing
**Forecasting: Principles & Practice (Python Edition)**
Lab Notebook · [github.com/bcseong2/fpppy-labs](https://github.com/bcseong2/fpppy-labs)

The cells below follow the slide deck *Ch8. Exponential smoothing* (updated, 54 slides) and are meant to be run from top to bottom: later cells reuse objects created in earlier ones.

In [ ]:
%pip install -q statsforecast utilsforecast matplotlib

In [ ]:
BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"  # datasets in this repository (data/)

## Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os; os.environ["NIXTLA_ID_AS_COL"] = "true"
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import AutoETS
plt.rcParams.update({"figure.figsize": (7, 3.5)})

## Data
The slides use these three datasets without showing how they are loaded; the code below follows the textbook (Sections 8.1, 8.2, 8.3).

In [ ]:
# Exports of goods and services from Algeria (% of GDP), 1960–2017
algeria_economy = pd.read_csv(BASE + "algeria_exports.csv", parse_dates=["ds"])

# Australian population (millions), 1960–2017
aus_economy = (
    pd.read_csv(BASE + "aus_economy.csv", parse_dates=["ds"])
    .assign(y=lambda x: x["y"] / 1e6)
)

# Australian domestic holiday trips (millions), quarterly 1998–2017
tourism = pd.read_csv(BASE + "tourism.csv", parse_dates=["ds"])
aus_holidays = (
    tourism.loc[lambda x: x["Purpose"] == "Holiday"]
    .rename(columns={"Purpose": "unique_id"})
    .groupby(["unique_id", "ds"])["y"].sum().div(1e3)
    .reset_index()
)

## [Slide 7] 8.1 Parameter Optimisation

In [ ]:
import numpy as np
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

sf = StatsForecast(
    models=[AutoETS(model="ANN", alias="SES")],
    freq="Y",
)
fc = sf.forecast(df=algeria_economy, h=5, level=[80, 95],
                  fitted=True)

## [Slide 8] 8.1 Parameter Optimisation (cont.)

In [ ]:
fitted_vals = sf.forecast_fitted_values()

# Extract optimal parameters
# (forecast() keeps no fitted models -> call fit())
sf.fit(df=algeria_economy)
ses = sf.fitted_[0, 0]
params = np.round(ses.model_["fit"].x, 4)
alpha, level_0 = params[0], params[1]
print(f"alpha = {alpha}, l0 = {level_0}")
# Example result: alpha = 0.84, l0 = 39.5

## [Slide 10] 8.1 Fitted Values, Residuals, Forecasts (1)

In [ ]:
y = algeria_economy["y"].to_numpy()
def make_sf():  # SES model used by all combinations
    return StatsForecast(
        models=[AutoETS(model="ANN", alias="SES")], freq="Y")

# 1. forecast(fitted=True): fit once, DataFrames
sf = make_sf()
fc1 = sf.forecast(df=algeria_economy, h=5, level=[80, 95],
                  fitted=True)                 # forecasts
fv1 = sf.forecast_fitted_values()              # fitted values
fv1["resid"] = fv1["y"] - fv1["SES"]           # residuals

# 2. fit() + predict(): fit once, parameters too
sf = make_sf().fit(df=algeria_economy)
fc2 = sf.predict(h=5, level=[80, 95])          # forecasts
m = sf.fitted_[0, 0]
fitted2 = m.predict_in_sample()["fitted"]      # fitted values
resid2 = y - fitted2                           # residuals

## [Slide 11] 8.1 Fitted Values, Residuals, Forecasts (2)

In [ ]:
# 3. fit_predict(): fit once, one call
sf = make_sf()
fc3 = sf.fit_predict(df=algeria_economy, h=5,
                     level=[80, 95])           # forecasts
m = sf.fitted_[0, 0]
fitted3 = m.model_["fitted"]                   # fitted values
resid3 = m.model_["residuals"]                 # residuals

# 4. forecast() + fit(): fits the model twice
sf = make_sf()
fc4 = sf.forecast(df=algeria_economy, h=5,
                  level=[80, 95])              # forecasts
sf.fit(df=algeria_economy)                     # second fit
fitted4 = sf.fitted_[0, 0].predict_in_sample()["fitted"]
resid4 = y - fitted4                           # residuals

## [Slide 13] 8.2 Holt's Linear Trend Method

In [ ]:
sf = StatsForecast(
    models=[AutoETS(model="AAN", alias="Holt")],
    freq="Y",
)
fc = sf.fit_predict(df=aus_economy, h=10)

## [Slide 14] 8.2 Holt's Linear Trend Method (cont.)

In [ ]:
holt = sf.fitted_[0, 0]
params = np.round(holt.model_["fit"].x, 4)
# Example: alpha = 0.9999, beta* = 0.2001

## [Slide 16] 8.2 Damped Trend Method

In [ ]:
AutoETS(model="AAN", damped=True, phi=0.9, alias="Damped")

## [Slide 19] 8.2 Internet Usage — Cross-Validation Code

In [ ]:
from functools import partial
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import rmse, mae, mase
from utilsforecast.losses import mape as _mape

def mape(df, models, **kw):          # MAPE in percent
    out = _mape(df, models, **kw)
    out[models] *= 100
    return out

www_usage = pd.read_csv(BASE + "www_usage.csv")
sf = StatsForecast(
    models=[AutoETS(model="ANN", alias="SES"),
            AutoETS(model="AAN", alias="Holt"),
            AutoETS(model="AAN", damped=True, phi=0.9,
                    alias="Damped")],
    freq=1)
cv = sf.cross_validation(df=www_usage, h=1, n_windows=90,
                         step_size=1)
acc = evaluate(cv.drop(columns="cutoff"),
               metrics=[rmse, mae, mape,
                        partial(mase, seasonality=1)],
               train_df=www_usage)

## [Slide 21] 8.2 Internet Usage — Final Model Code

In [ ]:
# phi is now estimated (not fixed at 0.9)
sf = StatsForecast(
    models=[AutoETS(model="AAN", damped=True,
                    alias="Damped")],
    freq=1)
sf.fit(df=www_usage)
m = sf.fitted_[0, 0].model_
print(m["method"])              # ETS(A,Ad,N)
print(np.round(m["fit"].x, 4))  # alpha, beta, phi, l0, b0
# [ 0.9999  0.2  0.9087  88.8688  -0.9231]

fc = sf.predict(h=10, level=[80, 95])

## [Slide 25] 8.3 Holt-Winters' Multiplicative Method (cont.)

In [ ]:
sf = StatsForecast(
    models=[
        AutoETS(season_length=4, model="AAA",
                alias="Additive"),
        AutoETS(season_length=4, model="MAM",
                alias="Multiplicative"),
    ],
    freq="Q",
)
fc = sf.forecast(df=aus_holidays, h=12, fitted=True)

## [Slide 27] 8.3 Holt-Winters Damped Method

In [ ]:
AutoETS(season_length=4, model="MAM", damped=True,
        alias="HW-Multiplicative-Damped")

## [Slide 41] 8.6 Automatic Model Selection: AIC (cont.)

In [ ]:
sf = StatsForecast(models=[AutoETS(season_length=4)],
                    freq="Q")
sf.fit(aus_holidays)
autoets = sf.fitted_[0, 0]
print(autoets.model_["method"])   # e.g. 'ETS(M,N,M)'
print(np.round(autoets.model_["fit"].x, 4))

## [Slide 44] 8.7 Point Forecasts from ETS

In [ ]:
sf = StatsForecast(
    models=[AutoETS(season_length=4)],
    freq="Q",
)
fc = sf.forecast(df=aus_holidays, h=8, level=[80, 95])

## [Slide 48] 8.7 Example: Visitor Nights in Australia

In [ ]:
tourism = pd.read_csv(BASE + "tourism.csv", parse_dates=["ds"])
aus_holidays = (
    tourism.loc[lambda x: x["Purpose"] == "Holiday"]
    .rename(columns={"Purpose": "unique_id"})
    .groupby(["unique_id", "ds"])["y"].sum().div(1e3)
    .reset_index()
)
sf = StatsForecast(models=[AutoETS(season_length=4)],
                   freq="QS")
sf.fit(aus_holidays)
autoets = sf.fitted_[0, 0]
model = autoets.model_
model["method"]                  # 'ETS(M,N,M)'
np.round(model["fit"].x, 4)
# alpha, gamma, l0, s0, s-1, s-2:
# [0.3612 0.0001 9.7878 0.9432 0.9268 0.9683]
np.round(model["states"][0], 4)
# initial states l0, s0, s-1, s-2, s-3:
# [9.7878 0.9432 0.9268 0.9683 1.1617]

## [Slide 49] 8.7 Example: Estimated Components — Code

In [ ]:
states = model["states"]           # row 0 = initial states
df = pd.DataFrame({
    "Trips": aus_holidays["y"].to_numpy(),
    "Level": states[1:, 0],         # l_t
    "Season": states[1:, 1],        # s_t
    "Residual": model["residuals"],
    "Quarter": aus_holidays["ds"].to_numpy(),
}).set_index("Quarter")
fig, axes = plt.subplots(4, 1, figsize=(8, 6.4), sharex=True)
for ax, column in zip(axes, df.columns):
    df[column].plot(ax=ax, color="black", lw=1)
    ax.set_ylabel(column)
axes[0].set_title("ETS(M,N,M) components", loc="left")
plt.show()

## [Slide 51] 8.7 Example: Residuals — Code

In [ ]:
df = pd.DataFrame({
    "Innovation residuals": model["residuals"],
    "Regular residuals": model["actual_residuals"],
    "Quarter": aus_holidays["ds"].to_numpy(),
}).set_index("Quarter")
fig, axes = plt.subplots(2, 1, figsize=(8, 4.4), sharex=True)
for ax, column in zip(axes, df.columns):
    df[column].plot(ax=ax, color="black", lw=1)
    ax.set_ylabel(column)
plt.show()

## [Slide 53] 8.7 Example: Forecasts — Code

In [ ]:
sf = StatsForecast(models=[AutoETS(season_length=4)],
                   freq="QS")
fc = sf.forecast(df=aus_holidays, h=8, level=[80, 95])

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(aus_holidays["ds"], aus_holidays["y"],
        color="black", lw=1, label="y")
for lv, a in [(95, .2), (80, .35)]:
    ax.fill_between(fc["ds"],
                    fc[f"AutoETS-lo-{lv}"],
                    fc[f"AutoETS-hi-{lv}"],
                    color="#2f2fff", alpha=a,
                    label=f"{lv}% PI")
ax.plot(fc["ds"], fc["AutoETS"],
        color="#2f2fff", label="AutoETS")
ax.set(xlabel="Quarter",
       ylabel="Overnight trips (millions)",
       title="Australian domestic tourism")
ax.legend(loc="upper left")
plt.show()